# Detect Domain-Specific Hallucinations in Your Chatbot

Catch invented numbers, omitted safety information, and entity confusion in domain-specific chatbots by grounding evaluations against your source documents.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/domain-hallucination-detection.ipynb)
[![GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/domain-hallucination-detection.ipynb)

| Time | Difficulty |
|------|------------|
| 30 min | Intermediate |

You have a domain-specific chatbot (medical, legal, financial, or similar) that answers questions using a RAG pipeline grounded in your source documents. Most of the time it works. But sometimes it invents numbers, omits critical information, or confuses similar-sounding entities. In high-stakes domains, these hallucinations are not just bad UX. They can cause real harm.

This cookbook shows you how to catch those hallucinations by grounding your evaluations against your actual source documents, running targeted RAG diagnostics, and building domain-specific eval rules that generic detectors miss.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- Python 3.9+
- Your domain source documents (PDF, TXT, DOCX, or RTF)

In [ ]:
!pip install futureagi ai-evaluation

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-api-key"
os.environ["FI_SECRET_KEY"] = "your-secret-key"

## Step 1: Upload your source documents as ground truth

Your source documents are the ground truth that every chatbot response must be traceable to. Upload them to a Knowledge Base so you can evaluate responses against them.

**From the dashboard:**

1. Go to [app.futureagi.com](https://app.futureagi.com) → **Knowledge base** (left sidebar) → **Create Knowledge Base**
2. Name it (e.g., `product-docs`, `drug-formulary`, `legal-briefs`)
3. Upload your source documents
4. Click **Create**

**From the SDK:**

In [ ]:
import os
from fi.kb import KnowledgeBase

kb_client = KnowledgeBase(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

kb_client.create_kb(
    name="my-domain-docs",
    file_paths=[
        "./docs/document-1.pdf",
        "./docs/document-2.pdf",
    ],
)

print(f"Knowledge Base created: {kb_client.kb.name}")

See [Knowledge Base](https://docs.futureagi.com/docs/cookbook/quickstart/knowledge-base) for managing files programmatically and adding documents to an existing KB.

## Step 2: Build test cases that mirror real failures

To demonstrate the evaluation pipeline, this cookbook uses a pharmaceutical chatbot as the running example. The same patterns apply to any domain: legal citations, financial regulations, product specifications, or internal policy documents.

Each test case has three parts: the user's question (`input`), the chatbot's response (`output`), and the retrieved source content (`context`). The hallucinations are deliberately the subtle kind: they sound plausible, which is exactly what makes them dangerous.

In [ ]:
test_cases = [
    {
        "input": "What is the recommended adult dose for ibuprofen, and what's the max daily OTC dose?",
        "context": (
            "Ibuprofen (Advil, Motrin): NSAID. Adult dose: 200-400 mg every 4-6 hours. "
            "Max daily dose: 1,200 mg (OTC) / 3,200 mg (prescription). "
            "Contraindications: active GI bleeding, severe renal impairment, third trimester pregnancy."
        ),
        "output": (
            "Ibuprofen is dosed at 200-400 mg every 4-6 hours for adults. "
            "The maximum OTC daily dose is 1,200 mg. For prescription use, the max is 3,200 mg per day. "
            "It's contraindicated in patients with active GI bleeding, severe renal impairment, "
            "or third trimester pregnancy."
        ),
        "label": "correct",
    },
    {
        "input": "What is the maximum daily dose of acetaminophen for an adult?",
        "context": (
            "Acetaminophen (Tylenol): Analgesic/antipyretic. Adult dose: 325-1,000 mg every 4-6 hours. "
            "Max daily dose: 4,000 mg (3,000 mg for patients with hepatic impairment or chronic "
            "alcohol use). Contraindications: severe hepatic impairment, active liver disease."
        ),
        "output": (
            "The maximum daily dose of acetaminophen for a healthy adult is 6,000 mg, taken as "
            "1,500 mg every 6 hours. For patients with liver conditions, reduce to 4,000 mg per day. "
            "Acetaminophen is very safe at therapeutic doses and rarely causes side effects."
        ),
        "label": "hallucinated: invented max dose of 6,000 mg (real limit is 4,000 mg)",
    },
    {
        "input": "What drug interactions should I watch for with lisinopril?",
        "context": (
            "Lisinopril (Zestril, Prinivil): ACE inhibitor. Drug interactions: potassium supplements "
            "and potassium-sparing diuretics (risk of hyperkalemia), NSAIDs (reduced antihypertensive "
            "effect), lithium (increased lithium levels)."
        ),
        "output": (
            "Lisinopril interacts with NSAIDs, which can reduce its blood pressure-lowering effect. "
            "It can also increase lithium levels if used concurrently. No significant interactions "
            "with potassium supplements have been reported."
        ),
        "label": "hallucinated: omits critical potassium interaction (hyperkalemia risk)",
    },
    {
        "input": "What are the common side effects of metformin?",
        "context": (
            "Metformin (Glucophage): Biguanide for type 2 diabetes. Common side effects: nausea, "
            "diarrhea, flatulence, vitamin B12 deficiency with long-term use. "
            "Contraindications: eGFR below 30 mL/min, metabolic acidosis, diabetic ketoacidosis."
        ),
        "output": (
            "Common side effects of metformin include nausea, diarrhea, flatulence, "
            "and vitamin B12 deficiency with long-term use. These GI side effects often improve "
            "with extended-release formulations or gradual dose titration."
        ),
        "label": "correct",
    },
    {
        "input": "Can a patient on warfarin take ibuprofen for pain?",
        "context": (
            "Ibuprofen (Advil, Motrin): NSAID. Drug interactions: increased bleeding risk with "
            "anticoagulants (warfarin), reduced efficacy of ACE inhibitors and ARBs, "
            "increased risk of GI bleeding with SSRIs."
        ),
        "output": (
            "Ibuprofen should be used with caution in patients on warfarin due to the increased "
            "risk of bleeding. Both drugs affect hemostasis: warfarin inhibits clotting factors "
            "while ibuprofen inhibits platelet aggregation and can cause GI irritation. "
            "Consider acetaminophen as a safer alternative for pain management."
        ),
        "label": "correct",
    },
    {
        "input": "What is the recommended starting dose and contraindications for metoprolol?",
        "context": (
            "Metformin (Glucophage): Biguanide for type 2 diabetes. Adult dose: 500 mg twice daily, "
            "titrated to max 2,550 mg/day. Contraindications: eGFR below 30 mL/min, metabolic "
            "acidosis, diabetic ketoacidosis."
        ),
        "output": (
            "Metoprolol is typically started at 500 mg twice daily for type 2 diabetes management. "
            "The maximum daily dose is 2,550 mg. It's contraindicated in patients with eGFR below "
            "30 mL/min, metabolic acidosis, and diabetic ketoacidosis."
        ),
        "label": "hallucinated: confused metformin with metoprolol (different drug class entirely)",
    },
]

Three hallucination patterns to catch:

- **Test case 2** invents a number. The source says 4,000 mg max; the chatbot says 6,000 mg.
- **Test case 3** omits critical information. The source lists a potassium interaction; the chatbot denies it exists.
- **Test case 6** confuses similar entities. The question asks about metoprolol, but the chatbot applies metformin data.

These are the three most common hallucination patterns in domain-specific chatbots, regardless of the domain.

## Step 3: Score each response against the source documents

Six evaluation metrics cover both layers of your RAG pipeline: retrieval quality (did the retriever fetch the right document?) and generation quality (did the LLM use that document correctly?). Running all six on each test case shows you where the failure started.

In [ ]:
from fi.evals import evaluate

for i, test in enumerate(test_cases):
    print(f"{'='*60}")
    print(f"Test case {i+1}: {test['input'][:60]}...")
    print(f"Label: {test['label'][:60]}...")
    print(f"{'='*60}\n")

    # --- Retrieval metrics ---

    # Did the retriever fetch the right source document?
    relevance = evaluate(
        "context_relevance",
        context=test["context"],
        input=test["input"],
        model="turing_small",
    )
    print(f"context_relevance  : score={relevance.score}  passed={relevance.passed}")
    print(f"  Reason: {relevance.reason}\n")

    # Can each claim be traced to a specific chunk?
    attribution = evaluate(
        "chunk_attribution",
        output=test["output"],
        context=test["context"],
        model="turing_small",
    )
    print(f"chunk_attribution  : score={attribution.score}  passed={attribution.passed}")
    print(f"  Reason: {attribution.reason}\n")

    # How much of the source content was actually used?
    utilization = evaluate(
        "chunk_utilization",
        output=test["output"],
        context=test["context"],
        model="turing_small",
    )
    print(f"chunk_utilization  : score={utilization.score}  passed={utilization.passed}")
    print(f"  Reason: {utilization.reason}\n")

    # --- Generation metrics ---

    # Is the response grounded in the source document?
    groundedness = evaluate(
        "groundedness",
        output=test["output"],
        input=test["input"],
        context=test["context"],
        model="turing_small",
    )
    print(f"groundedness       : score={groundedness.score}  passed={groundedness.passed}")
    print(f"  Reason: {groundedness.reason}\n")

    # Did the response fully answer the question?
    completeness = evaluate(
        "completeness",
        input=test["input"],
        output=test["output"],
        model="turing_small",
    )
    print(f"completeness       : score={completeness.score}  passed={completeness.passed}")
    print(f"  Reason: {completeness.reason}\n")

    # Are the stated facts correct given the source?
    accuracy = evaluate(
        "context_adherence",
        output=test["output"],
        context=test["context"],
        model="turing_small",
    )
    print(f"context_adherence   : score={accuracy.score}  passed={accuracy.passed}")
    print(f"  Reason: {accuracy.reason}\n")

For the correct responses (test cases 1, 4, 5), expect all metrics to pass. For the hallucinated responses, the interesting part is *which* metrics flag the problem. That tells you the root cause.

See [RAG Pipeline Evaluation](https://docs.futureagi.com/docs/cookbook/quickstart/rag-evaluation) for batch diagnostics and CI pipeline integration.

## Step 4: Identify whether retrieval or generation failed

Not all hallucinations have the same root cause. Some start at retrieval (wrong document fetched), others at generation (right document, wrong output). Knowing which layer broke tells you exactly where to apply the fix.

| Metric | What a failure means |
|---|---|
| `context_relevance` fails | Retriever fetched the wrong source document |
| `chunk_attribution` fails | Output contains claims that can't be traced to any source chunk |
| `chunk_utilization` fails | Chatbot ignored most of the retrieved content |
| `groundedness` fails | Response contains claims not in the source documents |
| `completeness` fails | Response doesn't fully answer the question |
| `context_adherence` fails | Stated facts are wrong given the source |

Use `context_relevance` and `groundedness` together to classify each failure:

In [ ]:
for i, test in enumerate(test_cases):
    relevance = evaluate(
        "context_relevance",
        context=test["context"],
        input=test["input"],
        model="turing_small",
    )
    groundedness = evaluate(
        "groundedness",
        output=test["output"],
        input=test["input"],
        context=test["context"],
        model="turing_small",
    )

    retrieval_ok = relevance.passed
    generation_ok = groundedness.passed

    if not retrieval_ok and not generation_ok:
        diagnosis = "Both retrieval and generation failing"
    elif not retrieval_ok:
        diagnosis = "RETRIEVAL problem: wrong source document fetched"
    elif not generation_ok:
        diagnosis = "GENERATION problem: LLM hallucinating despite correct source context"
    else:
        diagnosis = "Pipeline healthy"

    print(f"Test {i+1}: {diagnosis}")
    print(f"  Label: {test['label'][:60]}...\n")

What to expect for each hallucinated case:

- **Test case 2 (invented number):** `context_relevance` passes (right document fetched), but `groundedness` and `context_adherence` fail. This is a generation problem.
- **Test case 3 (omitted information):** `context_relevance` passes. `completeness` should flag the missing interaction. `context_adherence` should catch the false claim that "no significant interactions" exist.
- **Test case 6 (entity confusion):** `context_relevance` is the key. The retriever fetched metformin data for a metoprolol question. If relevance fails, the problem starts at retrieval.

See [Hallucination Detection](https://docs.futureagi.com/docs/cookbook/quickstart/hallucination-detection) for combining local NLI faithfulness checks with Turing-based groundedness scoring.

## Step 5: Add a custom eval for domain-specific rules

The built-in metrics catch general hallucination patterns. But your domain has rules that generic evaluators do not know. For example, a pharma chatbot must never state a dosage that does not appear in the formulary. A legal chatbot must never cite a statute that does not exist. A financial chatbot must never invent a fee schedule.

Create a custom eval with your domain rules.

**In the dashboard:**

1. Go to [app.futureagi.com](https://app.futureagi.com) → **Evals** (left sidebar under BUILD)
2. Click the **Create your own evals** card
3. **Step 1 (Add Details):** Enter name `domain_accuracy` (or something specific like `medication_accuracy`), select template type **Use Future AGI Agents**, then select language model `turing_small`
4. **Step 2 (Rule Prompt):** Paste your domain-specific rules. Here is an example for a pharma chatbot:

```
You are a domain accuracy checker for a chatbot grounded in source documents.

The chatbot's response: {{output}}
The source document content: {{context}}
The user's question: {{input}}

RULES (mark FAIL if ANY are violated):

1. NUMERICAL ACCURACY
   - All numbers (dosages, limits, thresholds) must exactly match the source
   - No invented, rounded, or extrapolated values

2. INFORMATION COMPLETENESS
   - All critical items listed in the source must be mentioned when asked
   - Omitting a listed item is a FAIL
   - Claiming something does not exist when the source says it does is a FAIL

3. ENTITY IDENTITY
   - The response must be about the correct entity
   - Applying Entity A's information to Entity B is a FAIL
   - Watch for similar-sounding names

4. SAFETY-CRITICAL OMISSIONS
   - Contraindications, warnings, and restrictions in the source must not be omitted
   - Downplaying a documented risk is a FAIL

Mark PASS only if every claim in the response is accurate according to the source,
no critical information is omitted, and the response is about the correct entity.

Return PASS or FAIL with a specific reason identifying which rule was violated.
```

5. **Step 3 (Output Type):** Select **Pass/Fail**
6. **Step 4 (Optional):** Add tags and description if needed
7. Click **Create Evaluation**

Now run it against your test cases:

In [ ]:
import os
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

for i, test in enumerate(test_cases):
    result = evaluator.evaluate(
        eval_templates="domain_accuracy",
        inputs={
            "output": test["output"],
            "context": test["context"],
            "input": test["input"],
        },
    )
    eval_result = result.eval_results[0]
    print(f"Test {i+1}: {eval_result.output}")
    print(f"  Reason: {eval_result.reason}\n")

The custom eval catches what generic metrics can miss:

- **Test case 2:** Fails on rule 1 (numerical accuracy). The source says 4,000 mg; the response says 6,000 mg.
- **Test case 3:** Fails on rule 2 (information completeness). The response claims no potassium interaction when the source explicitly lists one.
- **Test case 6:** Fails on rule 3 (entity identity). The response applies metformin data to metoprolol.

See [Custom Eval Metrics](https://docs.futureagi.com/docs/cookbook/quickstart/custom-eval-metrics) for creating evals with numerical scoring and running them on full datasets.

## Step 6: Upload test cases as a dataset and run batch evals with KB

For a production workflow, upload your test cases as a dataset and run evaluations with the Knowledge Base attached. This lets your team review results visually and track scores across iterations.

The `kb_id` parameter connects the evaluation to your Knowledge Base. FutureAGI uses the KB to ground its assessment of the chatbot's responses and retrieved context.

In [ ]:
import os
import csv
from fi.datasets import Dataset, DatasetConfig
from fi.utils.types import ModelTypes

# Write test cases to CSV
csv_path = "hallucination_test_data.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["input", "output", "context", "label"])
    writer.writeheader()
    for tc in test_cases:
        writer.writerow(tc)

# Upload to FutureAGI
dataset = Dataset(
    dataset_config=DatasetConfig(
        name="pharma-hallucination-tests",
        model_type=ModelTypes.GENERATIVE_LLM,
    ),
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)
dataset.create(source=csv_path)
print(f"Dataset created: {dataset.dataset_config.name}")

In [ ]:
from fi.kb import KnowledgeBase

# Fetch the existing KB by name
kb_client = KnowledgeBase(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
    kb_name="medisafe-drug-formulary",
)
kb_id = str(kb_client.kb.id)
print(f"Using KB: {kb_client.kb.name} (ID: {kb_id})")

# Run groundedness with KB (requires turing_large)
dataset.add_evaluation(
    name="groundedness-check",
    eval_template="groundedness",
    required_keys_to_column_names={
        "output": "output",
        "context": "context",
        "input": "input",
    },
    model="turing_large",
    run=True,
    reason_column=True,
    kb_id=kb_id,
)

# Run context_adherence with KB (requires turing_large)
dataset.add_evaluation(
    name="context-adherence-check",
    eval_template="context_adherence",
    required_keys_to_column_names={
        "input": "input",
        "output": "output",
        "context": "context",
    },
    model="turing_large",
    run=True,
    reason_column=True,
    kb_id=kb_id,
)

# Run completeness (no KB needed)
dataset.add_evaluation(
    name="completeness-check",
    eval_template="completeness",
    required_keys_to_column_names={
        "input": "input",
        "output": "output",
    },
    model="turing_small",
    run=True,
    reason_column=True,
)

print("All evaluations started. Check the dashboard for results.")


**View results in the dashboard:**

1. Go to **Dataset** (left sidebar) and click `pharma-hallucination-tests`
2. Once evaluations complete, you'll see new columns for each eval
3. Each row shows Pass/Fail and a reason explaining the verdict
4. The hallucinated rows (test cases 2, 3, 6) should show failures across groundedness, factual accuracy, and domain accuracy

## Step 7: Run the full diagnostic on every test case

At this point you have built-in RAG metrics and a custom domain eval. Running them together on the full test dataset gives you two signals per case: a general hallucination verdict and a domain-specific rule check. Together they leave nothing hidden.

In [ ]:
import os
from fi.evals import evaluate, Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

for i, test in enumerate(test_cases):
    print(f"\n{'='*60}")
    print(f"Test case {i+1}: {test['input'][:60]}...")
    print(f"Expected: {test['label'][:60]}...")
    print(f"{'='*60}")

    # Built-in RAG metrics
    groundedness = evaluate(
        "groundedness",
        output=test["output"],
        input=test["input"],
        context=test["context"],
        model="turing_small",
    )

    relevance = evaluate(
        "context_relevance",
        context=test["context"],
        input=test["input"],
        model="turing_small",
    )

    accuracy = evaluate(
        "context_adherence",
        output=test["output"],
        context=test["context"],
        model="turing_small",
    )

    completeness = evaluate(
        "completeness",
        input=test["input"],
        output=test["output"],
        model="turing_small",
    )

    # Custom domain eval
    domain_check = evaluator.evaluate(
        eval_templates="domain_accuracy",
        inputs={
            "output": test["output"],
            "context": test["context"],
            "input": test["input"],
        },
    )
    domain_result = domain_check.eval_results[0]

    # Summary
    print(f"  groundedness       : {'PASS' if groundedness.passed else 'FAIL'} (score={groundedness.score})")
    print(f"  context_relevance  : {'PASS' if relevance.passed else 'FAIL'} (score={relevance.score})")
    print(f"  context_adherence   : {'PASS' if accuracy.passed else 'FAIL'} (score={accuracy.score})")
    print(f"  completeness       : {'PASS' if completeness.passed else 'FAIL'} (score={completeness.score})")
    print(f"  domain_accuracy    : {domain_result.output}")

    if not groundedness.passed or not accuracy.passed:
        print(f"\n  Groundedness reason: {groundedness.reason}")
        print(f"  Accuracy reason: {accuracy.reason}")
        print(f"  Domain reason: {domain_result.reason}")

The built-in metrics tell you *something* is wrong. The custom eval tells you *exactly what* is wrong in your domain's terms. Together, they give you:

| Failure pattern | What it means | What to fix |
|---|---|---|
| `context_relevance` fails | Wrong source document retrieved | Improve chunking, add metadata filters, use exact-match retrieval for entity names |
| `groundedness` fails, `context_relevance` passes | LLM inventing facts despite having the right source | Tighten the system prompt: require verbatim numbers, forbid extrapolation |
| `completeness` fails | Critical information omitted | Add system prompt rules requiring complete listings when asked about interactions, warnings, etc. |
| `domain_accuracy` fails | Domain-specific rule violated | The custom eval reason tells you the exact rule and error |

## Step 8: Apply targeted fixes to retrieval and generation

The diagnostic output tells you which layer to fix. Different failures call for different remedies.

**When retrieval is the problem** (wrong source document fetched):

- Chunk by entity: each document, product, or topic should be its own chunk
- Add entity names as metadata and filter before passing to the LLM
- Use exact-match retrieval for entity names instead of relying solely on semantic similarity

**When generation is the problem** (right source, wrong output):

Update your system prompt with explicit constraints. Here is an example:

In [ ]:
SYSTEM_PROMPT = """You answer questions using ONLY the provided source context.

RULES:

1. Only state numbers, dates, and thresholds that appear verbatim in the context.
   Never round, estimate, or extrapolate.

2. When asked about lists (interactions, warnings, features), include ALL items
   from the context. Never omit any.

3. Verify that the entity in your response matches the entity in the question.
   If the context is about a different entity, say:
   "The available context is about [X], not [Y]. Let me clarify."

4. If the context does not contain sufficient information, say so explicitly.
   Never fill gaps with general knowledge.

Context: {context}

Question: {question}"""

print("System prompt created.")
print("After updating your pipeline with this prompt, re-run the full diagnostic")
print("from Step 6 on the same test cases to verify the fixes.")

After updating your pipeline, re-run the full diagnostic from Step 6 on the same test cases. The previously hallucinated scenarios should now produce grounded responses, or explicit "I don't have that information" fallbacks.

**Tip:** Run this diagnostic suite whenever you update your source documents. When content changes, your custom eval rules may need updating too. Treat your eval rules like your source documents: version them and review them regularly.

## What you solved

You can now detect domain-specific hallucinations in your chatbot by grounding evaluations against your source documents, diagnosing whether failures come from retrieval or generation, and applying targeted fixes.

- Uploaded source documents to a **Knowledge Base** as ground truth
- Built test cases covering the three most common hallucination patterns: invented numbers, omitted information, and entity confusion
- Ran **six RAG evaluation metrics** to diagnose each failure and classify it as a retrieval or generation problem
- Created a **custom eval** with domain-specific rules that generic detectors miss
- Combined built-in and custom evals in a **full diagnostic sweep**
- Applied **targeted fixes** to retrieval (entity-based chunking, metadata filters) and generation (constrained system prompt)

**Explore further:**
- [RAG Evaluation](https://docs.futureagi.com/docs/cookbook/quickstart/rag-evaluation): Debug retrieval vs generation with targeted metrics
- [Hallucination Detection](https://docs.futureagi.com/docs/cookbook/quickstart/hallucination-detection): Faithfulness, groundedness, and context adherence scoring
- [Custom Eval Metrics](https://docs.futureagi.com/docs/cookbook/quickstart/custom-eval-metrics): Write domain-specific evaluation criteria in plain English
- [Knowledge Base](https://docs.futureagi.com/docs/cookbook/quickstart/knowledge-base): Upload and manage documents for grounded evaluations